# Hybrid Search — Keyword + Semantic Retrieval

## Problem Statement
Build hybrid BM25 + semantic search with RRF fusion, and show it outperforms pure semantic search on exact-token queries.

## My Approach
Split retrieval into two independent passes - BM25 for exact token matching, ChromaDB for semantic similarity — then merge ranked results using Reciprocal Rank Fusion (RRF). RRF avoids score normalization by working on ranks instead of raw scores: each document gets `1/(k + rank)` from each list, and those are summed. Documents that appear in both lists get a double boost, which is exactly what we want for high-confidence matches.

## What I Learned
BM25 and semantic search fail in opposite directions - BM25 misses paraphrases, semantic search misses rare exact tokens. RRF is remarkably effective as a combiner because it doesn't require score normalization and naturally rewards consensus between the two rankers. The `k=60` constant acts as a smoothing factor - higher k means rank differences matter less.

## Where I Got Stuck
ChromaDB returns L2 distances (lower = more similar), not similarity scores - so my initial comparison table looked like the semantic search was performing poorly when it was actually spot on.

## What I'd Do Differently
in a real pipeline I'd store BM25 scores normalized per-query so I can weight the fusion more meaningfully - raw BM25 scores are not comparable across queries.

In [68]:
import chromadb
from rank_bm25 import BM25Okapi 
import numpy as np
import re

In [69]:
# Knowledge base 
RULES = [
    "Use `is not None` instead of `!= None` for None comparisons (PEP8 E711)",
    "Avoid single-letter variable names except in comprehensions (PEP8 E741)",
    "Never place import statements inside function bodies — put them at module top",
    "Use descriptive class names in PascalCase, not snake_case",
    "Prefer `enumerate()` over `range(len(list))` for indexed iteration",
    "Avoid mutable default arguments in function signatures",
    "Use context managers (`with`) for resource management like DB connections",
    "Keep functions under 20 lines — single responsibility principle",
    "Avoid bare `except:` clauses — catch specific exception types",
    "Use f-strings over `.format()` or `%` for string interpolation",
    
]

QUERIES = [
    "comparison with None using != instead of is not",
    "poor variable naming single letter",
    "import statement inside function body",
]

In [72]:
# BM25 index
tokenized = [re.findall(r"\b\w+\b", doc.lower()) for doc in RULES]
bm25 =  BM25Okapi(tokenized)

In [73]:
print(bm25.get_scores(QUERIES[0].split(" ")))

[5.97999773 0.         0.         1.30592215 0.         0.
 1.78535499 0.         0.         0.        ]


In [7]:
#ChromaDB semantic index
client=chromadb.PersistentClient("./chroma_db")
collection = client.get_or_create_collection("hybrid_search_test")
collection.add(
    documents=RULES,
    ids=[f"rule_{i}" for i in range(len(RULES))],
    metadatas=[{"type": "rule", "rule_id": i} for i in range(len(RULES))]
)

In [8]:
print(collection.count())

10


In [82]:
# BM25 index
def bm25_search(query: str, bm25 : BM25Okapi, top_k: int = 5): #-> list[tuple[int, float]]:
    # TODO: return [(doc_index, score), ...]
    bm25_scores=bm25.get_scores(query.split(" "))
    bm25_docranked= sorted(range(len(bm25_scores)) , key=lambda i : bm25_scores[i] , reverse=True) #this will give us index of rules
    result=[(doc_index , bm25_scores[doc_index] )  for i , (doc_index ) in  enumerate (bm25_docranked ) if (bm25_scores[doc_index] >0)]
    return result[:top_k]

In [83]:
#Test bm25_search
bm25_search(QUERIES[0],bm25)

[(0, np.float64(5.979997734568627)),
 (6, np.float64(1.7853549892495557)),
 (3, np.float64(1.3059221473420166))]

In [84]:
def semantic_search(query: str, collection, top_k: int = 5) -> list[tuple[int, float]]:
    query_results=collection.query( query_texts=[query] , n_results=top_k)
    result = [(m["rule_id"], round(1 / (1 + d), 4)) for m, d in zip(query_results["metadatas"][0], query_results["distances"][0])]  
    return result

In [85]:
#Test semantic_search
semantic_search(QUERIES[0] , collection)


[(0, 0.7432), (8, 0.4096), (1, 0.3831), (9, 0.3652), (3, 0.36)]

In [86]:
def rrf_fusion(bm25_results, semantic_results, k: int = 60, weights: tuple = (1.0, 1.0)) -> list[tuple[int, float]]:
    bm25_ranks = {doc_id: rank for rank, (doc_id, _) in enumerate(bm25_results, 1)}
    semantic_ranks = {doc_id: rank for rank, (doc_id, _) in enumerate(semantic_results, 1)}
    
    all_doc_ids = set(bm25_ranks.keys()) | set(semantic_ranks.keys())
    
    rrf_scores = []
    for doc_id in all_doc_ids:
        rank_bm25 = bm25_ranks.get(doc_id)
        score_bm25 = weights[0] * (1 / (k + rank_bm25)) if rank_bm25 else 0
        
        rank_sem = semantic_ranks.get(doc_id)
        score_sem = weights[1] * (1 / (k + rank_sem)) if rank_sem else 0
        
        rrf_scores.append((doc_id, score_bm25 + score_sem))
    
    return sorted(rrf_scores, key=lambda x: x[1], reverse=True)


In [87]:
#Test rrf_fusion
rrf_fusion(bm25_search(QUERIES[0],bm25),semantic_search(QUERIES[0] , collection) , weights=(1,1))

[(0, 0.03278688524590164),
 (3, 0.03125763125763126),
 (6, 0.016129032258064516),
 (8, 0.016129032258064516),
 (1, 0.015873015873015872),
 (9, 0.015625)]

In [88]:
# Side-by-side comparison
TOP_K = 3
DIVIDER = "-" * 70

for query in QUERIES:
    bm25_res    = bm25_search(query, bm25, top_k=TOP_K)
    sem_res     = semantic_search(query, collection, top_k=TOP_K)
    hybrid_res  = rrf_fusion(bm25_res, sem_res)[:TOP_K]

    print(DIVIDER)
    print(f'Query: "{query}"')
    print()

    print("  --- Pure BM25 Top 3 ---")
    for rank, (idx, score) in enumerate(bm25_res, 1):
        print(f"  [{rank}] Rule {idx:2d} (score: {score:.3f}): {RULES[idx][:70]}")
    if not bm25_res:
        print("  [no keyword matches]")

    print()
    print("  --- Pure Semantic Top 3 ---")
    for rank, (idx, score) in enumerate(sem_res, 1):
        print(f"  [{rank}] Rule {idx:2d} (sim: {score:.4f}): {RULES[idx][:70]}")

    print()
    print("  --- Hybrid (RRF) Top 3 ---")
    for rank, (idx, score) in enumerate(hybrid_res, 1):
        print(f"  [{rank}] Rule {idx:2d} (rrf: {score:.6f}): {RULES[idx][:70]}")

    print()
    # Winner logic: hybrid wins if top-1 matches expected (BM25 exact hit) and differs from semantic
    hybrid_top  = hybrid_res[0][0]  if hybrid_res  else None
    sem_top     = sem_res[0][0]     if sem_res      else None
    bm25_top    = bm25_res[0][0]    if bm25_res     else None

    if bm25_top is not None and bm25_top == hybrid_top and bm25_top != sem_top:
        verdict = "HYBRID  — BM25 exact match promoted a rule semantic missed"
    elif hybrid_top == sem_top == bm25_top:
        verdict = "ALL AGREE — strong signal, all three methods converge on same top rule"
    else:
        verdict = "SEMANTIC  — semantic alone was sufficient for this query"
    print(f"  Winner: {verdict}")

print(DIVIDER)

----------------------------------------------------------------------
Query: "comparison with None using != instead of is not"

  --- Pure BM25 Top 3 ---
  [1] Rule  0 (score: 5.980): Use `is not None` instead of `!= None` for None comparisons (PEP8 E711
  [2] Rule  6 (score: 1.785): Use context managers (`with`) for resource management like DB connecti
  [3] Rule  3 (score: 1.306): Use descriptive class names in PascalCase, not snake_case

  --- Pure Semantic Top 3 ---
  [1] Rule  0 (sim: 0.7432): Use `is not None` instead of `!= None` for None comparisons (PEP8 E711
  [2] Rule  8 (sim: 0.4096): Avoid bare `except:` clauses — catch specific exception types
  [3] Rule  1 (sim: 0.3831): Avoid single-letter variable names except in comprehensions (PEP8 E741

  --- Hybrid (RRF) Top 3 ---
  [1] Rule  0 (rrf: 0.032787): Use `is not None` instead of `!= None` for None comparisons (PEP8 E711
  [2] Rule  6 (rrf: 0.016129): Use context managers (`with`) for resource management like DB connecti